# HyperKT Objective 01 — Stage 3: Differentiable Hypergraph Learning
**Runtime:** Google Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU). CPU fallback supported.  
**Input:** `hyperkt_stage2.zip` (uploaded in Colab). **Output:** `hyperkt_stage3.zip`.  
This stage trains **only the structural hypergraph** on training-split co-occurrence; it does **not** produce knowledge-tracing AUC/F1. Results must not be reported as predictive performance.

**Paper consistency note:** The manuscript's `A_prob @ W_e @ A_prob.T` has incompatible dimensions if `W_e` is `n_edges × hidden`. Here a mathematically valid symmetric reconstruction is `sigmoid((A_prob * edge_strength) @ A_prob.T * scale + bias)`. This is a documented correction requiring manuscript alignment. No direction/prerequisite causality is inferred from co-occurrence.

In [ ]:
import os, io, json, zipfile, random, hashlib
import numpy as np, scipy.sparse as sp, pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from google.colab import files
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:',torch.__version__,'Device:',device)
if device.type=='cuda': print('GPU:',torch.cuda.get_device_name(0))

In [ ]:
uploaded=files.upload()
assert 'hyperkt_stage2.zip' in uploaded, 'Upload hyperkt_stage2.zip with exactly that filename'
raw=uploaded['hyperkt_stage2.zip']; source_sha=hashlib.sha256(raw).hexdigest()
z=zipfile.ZipFile(io.BytesIO(raw))
required={'train_cooccurrence.npz','train_sequences.npz','stage2_report.json','manifest.json','skill_vocab.json'}
assert required.issubset(z.namelist()), required-set(z.namelist())
report=json.loads(z.read('stage2_report.json'))
print('Stage 2 report:',json.dumps(report,indent=2)[:1800])

In [ ]:
co=sp.load_npz(io.BytesIO(z.read('train_cooccurrence.npz'))).tocsr()
assert co.shape[0]==co.shape[1] and np.isfinite(co.data).all()
print('Co-occurrence matrix:',co.shape,'nnz:',co.nnz,'range:',(co.data.min(),co.data.max()))
print('Symmetry max diff:',abs(co-co.T).max() if (co-co.T).nnz else 0)
print('Diagonal sample:',co.diagonal()[:10])
seq=np.load(io.BytesIO(z.read('train_sequences.npz')))
for k in seq.files: print(k,seq[k].shape)
assert np.all(seq['target_skill'][seq['mask']]>=2), 'Expected known skills >=2 in valid targets'
print('Known skill ID maximum:',seq['target_skill'][seq['mask']].max())

## Matrix alignment and normalization
Stage 1 reserves skill IDs 0 and 1 for unknown and padding. The co-occurrence matrix may use a different indexing convention. The following explicit check accepts either an `n_known × n_known` matrix or an `n_known+2 × n_known+2` matrix. It fails rather than guessing on any other shape.

In [ ]:
vocab=json.loads(z.read('skill_vocab.json'))
print('Vocabulary format:',type(vocab).__name__,'preview:',str(vocab)[:350])
known_ids=np.unique(np.concatenate([seq['history_skill'][seq['mask']],seq['target_skill'][seq['mask']]]))
known_ids=known_ids[known_ids>=2]
max_skill=int(max(known_ids))
if co.shape[0]==max_skill-1:
    skill_ids=np.arange(2,max_skill+1,dtype=int)
elif co.shape[0]==max_skill+1:
    skill_ids=np.arange(2,max_skill+1,dtype=int)
    co=co[2:,2:].tocsr()
else:
    raise ValueError(f'Cannot safely align co-occurrence size {co.shape} with max encoded skill ID {max_skill}; inspect Stage 1 mapping')
assert co.shape[0]==len(skill_ids)
print('Aligned skills:',len(skill_ids),'IDs:',skill_ids[0],skill_ids[-1])
G=co.toarray().astype(np.float32)
assert np.isfinite(G).all() and G.min()>=0 and G.max()<=1+1e-5
G=np.clip((G+G.T)/2,0,1)
np.fill_diagonal(G,0)  # no trivial self-co-occurrence objective
print('Off-diagonal positive rate:',np.mean(G[np.triu_indices(len(G),1)]>0))

## Learnable hypergraph
Softmax memberships sum to one across hyperedges. To prevent a degenerate constant reconstruction, the loss combines off-diagonal MSE, a weighted positive-pair term, and a mild entropy regularizer. Hyperedge-strength L1 is used as a sparsity proxy; unlike softmax membership L1, it is **not constant**. We log each component.

In [ ]:
class DifferentiableHypergraph(nn.Module):
    def __init__(self,n_skills,n_edges=32):
        super().__init__()
        self.logits=nn.Parameter(torch.randn(n_skills,n_edges)*0.05)
        self.raw_strength=nn.Parameter(torch.zeros(n_edges))
        self.scale=nn.Parameter(torch.tensor(1.0))
        self.bias=nn.Parameter(torch.tensor(-2.0))
    def forward(self):
        A=torch.softmax(self.logits,dim=1)
        strengths=F.softplus(self.raw_strength)
        logits=(A*strengths)@A.T * F.softplus(self.scale) + self.bias
        return torch.sigmoid(logits),A,strengths

model=DifferentiableHypergraph(len(skill_ids),n_edges=min(32,len(skill_ids))).to(device)
target=torch.tensor(G,device=device)
upper=torch.triu(torch.ones_like(target,dtype=torch.bool),diagonal=1)
y=target[upper]
print('Pairs:',y.numel(),'Nonzero pairs:',(y>0).sum().item())

In [ ]:
EPOCHS=500; LR=0.02; PATIENCE=60; LAMBDA_SPARSE=0.01
opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
best=float('inf'); best_state=None; wait=0; rows=[]
for epoch in range(1,EPOCHS+1):
    model.train(); opt.zero_grad()
    recon,A,strength=model()
    p=recon[upper]
    mse=F.mse_loss(p,y)
    pos=(y>0)
    pos_mse=F.mse_loss(p[pos],y[pos]) if pos.any() else mse
    sparse=strength.mean()
    loss=mse+pos_mse+LAMBDA_SPARSE*sparse
    loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
    rows.append(dict(epoch=epoch,loss=loss.item(),mse=mse.item(),positive_mse=pos_mse.item(),sparsity=sparse.item()))
    if loss.item()<best-1e-7:
        best=loss.item();best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()};wait=0
    else:wait+=1
    if epoch==1 or epoch%50==0:print(f'epoch {epoch:03d} loss {loss.item():.6f} mse {mse.item():.6f} pos {pos_mse.item():.6f}')
    if wait>=PATIENCE:print('Early stop:',epoch);break
model.load_state_dict(best_state);model.eval()
with torch.no_grad(): recon,A,strength=model()
A=A.cpu().numpy(); strengths=strength.cpu().numpy(); R=recon.cpu().numpy()
print('Best training reconstruction loss:',best)
print('Reconstruction MSE (off diagonal):',np.mean((R[np.triu_indices(len(G),1)]-G[np.triu_indices(len(G),1)])**2))

In [ ]:
import matplotlib.pyplot as plt
history=pd.DataFrame(rows)
plt.figure(figsize=(7,4));plt.plot(history.epoch,history.loss,label='total');plt.plot(history.epoch,history.mse,label='MSE');plt.xlabel('Epoch');plt.ylabel('Training loss');plt.legend();plt.tight_layout();plt.savefig('stage3_loss.png',dpi=180);plt.show()
plt.figure(figsize=(8,4));plt.imshow(A,aspect='auto',interpolation='nearest');plt.colorbar(label='Assignment probability');plt.xlabel('Hyperedge');plt.ylabel('Skill index');plt.tight_layout();plt.savefig('stage3_membership.png',dpi=180);plt.show()

In [ ]:
top=[]
for e in range(A.shape[1]):
    order=np.argsort(A[:,e])[::-1][:8]
    top.append({'hyperedge':e,'strength':float(strengths[e]),'top_skill_ids':','.join(map(str,skill_ids[order])),'top_probabilities':','.join(f'{A[i,e]:.4f}' for i in order)})
topdf=pd.DataFrame(top).sort_values('strength',ascending=False)
display(topdf.head(12))
np.savez_compressed('stage3_hypergraph.npz',skill_ids=skill_ids,assignments=A,edge_strengths=strengths,reconstruction=R,observed=G)
torch.save({'model_state_dict':best_state,'num_skills':len(skill_ids),'num_edges':A.shape[1],'skill_ids':skill_ids.tolist(),'seed':SEED},'stage3_hypergraph.pt')
history.to_csv('stage3_training_history.csv',index=False)
topdf.to_csv('stage3_top_hyperedges.csv',index=False)
summary={'stage':3,'seed':SEED,'device':str(device),'source_stage2_sha256':source_sha,'num_skills':len(skill_ids),'num_hyperedges':A.shape[1],'epochs_trained':len(history),'best_train_loss':best,'off_diagonal_reconstruction_mse':float(np.mean((R[np.triu_indices(len(G),1)]-G[np.triu_indices(len(G),1)])**2)),'caveat':'Training-only structural fit; not a knowledge-tracing test metric or causal prerequisite validation'}
with open('stage3_report.json','w') as f:json.dump(summary,f,indent=2)
with zipfile.ZipFile('hyperkt_stage3.zip','w',zipfile.ZIP_DEFLATED) as out:
    for name in ['stage3_hypergraph.npz','stage3_hypergraph.pt','stage3_training_history.csv','stage3_top_hyperedges.csv','stage3_loss.png','stage3_membership.png','stage3_report.json']:
        out.write(name)
print(json.dumps(summary,indent=2));files.download('hyperkt_stage3.zip')